<a href="https://colab.research.google.com/github/muneer-ahmad10/End_Module_exam_prep/blob/main/weak_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **The RAG Pipeline**

***one continuous story: raw PDF → chunks → embeddings → vector index → retriever → reader/generator → orchestration → full app.***

# **Pretrained Transformers for Embedding Generation & Fine-tuning**

### **Word2Vec-style embeddings**

one fixed vector per word, regardless of context ("bank" always got the same vector, whether river bank or money bank). Transformer-based embeddings (from BERT-style encoders) are contextual — the same word gets a different vector depending on the sentence it's in, because self-attention lets each token's representation absorb information from surrounding words.

**Sentence embeddings:** to represent a whole sentence/document (not just one word) as a single vector, you typically pool the token-level outputs — commonly by taking the [CLS] token's representation (Day 10-13 callback), or averaging all token embeddings (mean pooling). This single vector is what later gets stored and searched in Day 26's FAISS index

**Fine-tuning embeddings:** pretrained embedding models can be fine-tuned on domain-specific data (e.g. legal or medical text) so that "similar meaning" is defined in a way that matches your specific domain, rather than generic web text.

## **PDF Processing and Chunk-Level QA**

Before any embedding/search happens, you need raw text out of a PDF:

* Extract text from the PDF (handling multi-column layouts, tables, headers/footers as noise to potentially strip)

* Split ("chunk") the text into smaller pieces — because a full PDF is far too long to fit into a model's context window at once, and because retrieval works better on focused, topically-coherent pieces

**Chunk-Level QA:** given a single chunk of text and a question, extract the answer from that chunk specifically — this is the simplest form of the QA task, before you introduce the complexity of "which chunk even has the answer?"

## **Semantic-Aware Chunking and Semantic Embedding**

### **The problem with naive chunking**

The simplest chunking approach just splits text every N characters/words (e.g. every 500 words), regardless of where sentences or ideas actually end. This can slice a sentence or idea in half mid-thought — a chunk might end with "...the treatment showed significant" and the next chunk starts with "improvement in patients who...", breaking the semantic unit apart right at the boundary.

### **Semantic-aware chunking — the fix**

Instead of splitting by a fixed character/word count blindly, semantic chunking tries to split at natural topic/meaning boundaries — e.g. splitting by paragraph, by detecting where sentence embeddings shift significantly in meaning (indicating a topic change), or using sentence boundaries as safer split points rather than cutting mid-sentence.

**Common practical strategies:**

* **Recursive/hierarchical splitting**: try to split by paragraph first; if a paragraph is still too long, fall back to splitting by sentence; if a sentence is still too long, fall back to splitting by fixed size — always preferring the most "natural" boundary available

* **Overlap between chunks**: even with good boundaries, adding a small overlap (e.g. 50 words) between consecutive chunks helps preserve context that might otherwise be lost right at a boundary

* **Embedding-based splitting**: compute embeddings for consecutive sentences, and split where the similarity between adjacent sentence embeddings drops sharply — signaling a topic shift